# Make slim `model.pth` for submission zip

학습 ckpt(`ckpt_*.pt`, ~830 MB)에서 **Generator EMA 가중치 + meta**만 추출해
`checkpoints/model.pth`(~120 MB)로 저장.

- 학습용 ckpt: G/D/EMA/optG/optD/RNG/pl_mean/wandb_run_id/meta
- 제출용 model.pth: **G_ema_state + meta.generator_config + images_seen** 만
- generate.py / export_onnx.py와 호환 (둘 다 ckpt['G_ema_state'] + ckpt['meta']['generator_config']을 읽음)
- PDF spec: "Do not need to submit the discriminator"

In [ ]:
# 1. Drive mount
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. 경로 설정
DRIVE   = '/content/drive/MyDrive/osai/p2'
RUN_DIR = f'{DRIVE}/runs/d256_main'
DST     = f'{DRIVE}/checkpoints/model.pth'

import os
os.makedirs(os.path.dirname(DST), exist_ok=True)

In [ ]:
# 3. 최신 ckpt 자동 탐색 (final.pt 우선, 없으면 가장 큰 ckpt_*.pt)
import glob, os
candidates = []
if os.path.exists(f'{RUN_DIR}/final.pt'):
    candidates.append(f'{RUN_DIR}/final.pt')
candidates += sorted(glob.glob(f'{RUN_DIR}/ckpt_*.pt'))
assert candidates, f'no ckpt found in {RUN_DIR}'
SRC = candidates[-1]
print(f'Source : {SRC}')
print(f'  size : {os.path.getsize(SRC)/1e6:.2f} MB')

In [ ]:
# 4. Slim 추출 + 저장
import torch

ckpt = torch.load(SRC, map_location='cpu', weights_only=False)
print('Source keys:', sorted(ckpt.keys()))

slim = {
    'G_ema_state': ckpt['G_ema_state'],
    'meta': {
        'generator_config': ckpt['meta']['generator_config'],
    },
    'images_seen': ckpt.get('images_seen', 0),
}

torch.save(slim, DST)
print(f'\nSaved : {DST}')
print(f'  size : {os.path.getsize(DST)/1e6:.2f} MB')

In [ ]:
# 5. Sanity — 다시 로드해서 dict 구조와 텐서 크기 확인
import torch
loaded = torch.load(DST, map_location='cpu', weights_only=False)
print('Loaded keys :', sorted(loaded.keys()))
print(f'images_seen  : {loaded["images_seen"]:,}')
print(f'meta.generator_config :')
for k, v in loaded['meta']['generator_config'].items():
    print(f'  {k} : {v}')
g_state = loaded['G_ema_state']
print(f'\nG_ema_state : {len(g_state)} parameter tensors')
total_p = sum(t.numel() for t in g_state.values())
print(f'  total params : {total_p/1e6:.4f}M  (expected ~30.04M)')
print(f'\nFirst 3 keys + shapes:')
for k in list(g_state.keys())[:3]:
    print(f'  {k} : {tuple(g_state[k].shape)}')

## 결과

위 출력에서 다음 확인:
- `Loaded keys` = `['G_ema_state', 'images_seen', 'meta']`
- `total params` ≈ **30.04M** (Generator 한 마리)
- 파일 크기 ~**120 MB**

이 `model.pth`가 제출 zip의 `checkpoints/model.pth` 위치에 들어갈 파일입니다.

**호환성**: 학습 코드의 `generate.py --ckpt model.pth`와 `export_onnx.py --ckpt model.pth` 둘 다 그대로 동작 (둘 다 `ckpt['G_ema_state']` + `ckpt['meta']['generator_config']`만 읽음).